# Backward pass of a softmax classifier

A softmax classifier is one linear layer followed by a softmax and the mean cross-entropy
loss. You write its forward pass in a form that cannot overflow, a finite-difference checker
for arrays of any shape, and the gradients of the loss with respect to the weights and the
bias for a whole batch, in NumPy alone.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# Twelve examples with four features, three classes.
rng = np.random.default_rng(3)
N, d, C = 12, 4, 3
X = rng.standard_normal((N, d))
y = rng.integers(0, C, size=N)
W = 0.5 * rng.standard_normal((C, d))
b = np.array([0.1, -0.2, 0.0])
print("labels:", y)

## Part 1 (coding task): A softmax that cannot overflow

Write `softmax(Z)` for a 2-D array of logits, one example per row. It returns an array of
the same shape whose row $n$ is the softmax of row $n$ of `Z`. Subtract each row's own
maximum before exponentiating, so that no exponential overflows and no row underflows to all
zeros.

In [ ]:
def softmax(Z):
    """Row-wise softmax of a 2-D array."""
    # TODO
    raise NotImplementedError

In [ ]:
P = softmax(np.array([[np.log(2.0), 0.0, 0.0], [0.0, np.log(3.0), np.log(6.0)]]))
assert P.shape == (2, 3), f"expected shape (2, 3), got {P.shape}"
assert np.allclose(P.sum(axis=1), 1), "every row should add up to 1: normalize along axis 1, within each example"
assert np.allclose(P, [[0.5, 0.25, 0.25], [0.1, 0.3, 0.6]]), f"wrong probabilities:\n{P}"
with np.errstate(all="ignore"):
    big = softmax(np.array([[1000.0, 1000.0, 999.0], [-1000.0, -1003.5, -998.0]]))
assert not np.isnan(big).any(), "nan for large logits: subtract the maximum of each row before exponentiating (one maximum for the whole array is not enough when rows sit far apart)"
e1, e2 = np.exp([0.0, 0.0, -1.0]), np.exp([-2.0, -5.5, 0.0])
assert np.allclose(big, [e1 / e1.sum(), e2 / e2.sum()]), "wrong probabilities for large logits"
print("Part 1 passed")

## Part 2 (coding task): A finite-difference checker for any array

Write `numeric_grad(f, A, h=1e-5)`. Here `f` is a function that reads the array `A` and returns
a number, and the result is an array shaped like `A` whose every entry is the central difference

$$
\frac{f(A + h E) - f(A - h E)}{2h},
$$

where $E$ is 1 at that entry and 0 elsewhere. `A` may have any number of dimensions. When your
function returns, `A` must hold exactly the values it had before. A loop is fine here.

A hint, if you want one: `np.ndindex(A.shape)` visits every index of an array of any shape.

In [ ]:
def numeric_grad(f, A, h=1e-5):
    """Central-difference gradient of f with respect to every entry of A."""
    # TODO
    raise NotImplementedError

In [ ]:
grid = np.array([[0.5, -1.0, 2.0], [1.5, 0.0, -0.3]])
saved = grid.copy()
g = numeric_grad(lambda M: np.sum(np.sin(M) * M), grid)
assert (grid == saved).all(), "A was changed: restore every entry after nudging it"
assert np.shape(g) == grid.shape, f"the result should have the shape of A, {grid.shape}, got {np.shape(g)}"
exact = np.cos(grid) * grid + np.sin(grid)
assert np.allclose(g, exact, atol=1e-4), f"wrong values:\n{np.round(g, 6)}\nexpected\n{np.round(exact, 6)}"
assert np.allclose(g, exact, atol=1e-8), "close but not accurate enough: use the central difference (f(A + hE) - f(A - hE)) / 2h, whose error shrinks like h squared"
v = np.array([1.0, 2.0, 3.0])
assert np.allclose(numeric_grad(lambda u: np.sum(u ** 2), v), 2 * v, atol=1e-8), "wrong on a 1-D array"
print("Part 2 passed")

## Part 3 (coding task): The loss and its gradients for a batch

Write `batch_gradients(W, b, X, labels)` for the classifier

$$
Z = X W^\top + \mathbf{b}, \qquad P = \operatorname{softmax}(Z), \qquad
\mathcal{L} = -\frac1N\sum_{n=1}^{N} \ln P_{n, y_n},
$$

where `X` is $N \times d$, `W` is $C \times d$, `b` has length $C$ and `labels` holds the true
class $y_n$ of each row as an integer. It returns `(loss, dW, db)` with `dW` shaped like `W`
and `db` shaped like `b`. Use your `softmax` and no loop over the examples. Two results from
the lesson are all you need: for one example the loss's gradient in its logits is
prediction minus target, with the target one-hot, and a layer's weight gradient is
an outer product.

In [ ]:
def batch_gradients(W, b, X, labels):
    """Mean cross-entropy of a softmax classifier, and its gradients for W and b."""
    # TODO
    raise NotImplementedError

In [ ]:
loss, dW, db = batch_gradients(W, b, X, y)
Z = X.dot(W.transpose()) + b
lse = np.log(np.exp(Z).sum(axis=1))
assert np.isclose(loss, np.mean(lse - Z[np.arange(N), y])), f"the loss {loss} is wrong: average -log P[n, y_n] over the batch"
dW, db = np.asarray(dW, dtype=float), np.asarray(db, dtype=float)
assert dW.shape == W.shape, f"dW should have the shape of W, {W.shape}, got {dW.shape}"
assert db.shape == b.shape, f"db should have the shape of b, {b.shape}, got {db.shape}"

def mean_ce(W_, b_):
    Z_ = X.dot(W_.transpose()) + b_
    return np.mean(np.log(np.exp(Z_).sum(axis=1)) - Z_[np.arange(N), y])

# Your checker from Part 2, on copies so that W and b stay as they are.
dW_num = numeric_grad(lambda M: mean_ce(M, b), W.copy())
db_num = numeric_grad(lambda v: mean_ce(W, v), b.copy())
assert not np.allclose(dW, N * dW_num, atol=1e-5), "dW is N times too large: the loss is a mean over the batch"
assert np.allclose(dW, dW_num, atol=1e-6), f"dW does not match finite differences:\n{np.round(dW, 4)}\nexpected\n{np.round(dW_num, 4)}"
assert np.allclose(db, db_num, atol=1e-6), f"db does not match finite differences: {np.round(db, 4)}, expected {np.round(db_num, 4)}"
with np.errstate(all="ignore"):
    _, dW_big, db_big = batch_gradients(400 * W, b, X, y)
assert np.isfinite(dW_big).all() and np.isfinite(db_big).all(), "nan or inf for large weights: compute the gradients from your stable softmax"
assert np.allclose(db.sum(), 0), "the entries of db should add up to 0, since each row of P - Y does"
print("Part 3 passed")

## Solution

### Part 1: A softmax that cannot overflow

In [ ]:
def softmax(Z):
    """Row-wise softmax of a 2-D array."""
    shifted = np.asarray(Z, dtype=float) - np.max(Z, axis=1, keepdims=True)
    weights = np.exp(shifted)                  # every exponent is at most 0
    return weights / np.sum(weights, axis=1, keepdims=True)

**Why it works.** Subtracting a row's maximum multiplies every exponential in that row by the same factor,
which cancels in the division, so the probabilities do not change. Afterwards the largest
exponent in each row is $e^0 = 1$, so nothing overflows and every denominator is at least 1.
Subtracting one maximum for the whole array is not enough: the second test row sits
about 2000 below it, every exponential in the row would underflow to 0, and the row would become
$0/0$.

### Part 2: A finite-difference checker for any array

In [ ]:
def numeric_grad(f, A, h=1e-5):
    """Central-difference gradient of f with respect to every entry of A."""
    grad = np.zeros_like(A, dtype=float)
    for idx in np.ndindex(A.shape):
        old = A[idx]
        A[idx] = old + h
        up = f(A)
        A[idx] = old - h
        down = f(A)
        A[idx] = old                       # put the entry back
        grad[idx] = (up - down) / (2 * h)
    return grad

**Why it works.** Each entry is nudged up and down by $h$ with the others held fixed, which is exactly the partial
derivative's recipe. Writing into `A` in place and putting the old value back keeps the
function cheap, since nothing is copied, and leaves `A` as it was, which matters when `A` is
a model's own weight array. The one-sided difference would pass the first value test and fail
the second: its error is about $h$ times the second derivative, near $10^{-5}$, against about
$h^2$, near $10^{-10}$, for the central one.

### Part 3: The loss and its gradients for a batch

In [ ]:
def batch_gradients(W, b, X, labels):
    """Mean cross-entropy of a softmax classifier, and its gradients for W and b."""
    rows = np.arange(len(labels))
    probs = softmax(X @ W.T + b)                # (N, C)
    loss = -np.log(probs[rows, labels]).mean()
    delta = probs.copy()
    delta[rows, labels] -= 1.0                  # prediction minus one-hot target
    delta /= len(labels)                        # the loss is a mean over the batch
    dW = delta.T @ X                            # sum of outer products, (C, d)
    db = delta.sum(axis=0)                      # b is added to every row, (C,)
    return loss, dW, db

**Why it works.** Row $n$ of $P - Y$ is example $n$'s gradient in its own logits, and dividing by $N$ makes it the
gradient of the mean. The weights are shared by every example, so the layer rule's outer
products $(\mathbf{p}_n - \mathbf{y}_n)\,\mathbf{x}_n^\top$ add up over the batch, which is
$G^\top X$, of shape $C \times d$ like $W$. The bias is added to every row of $Z$, so its gradient
sums the rows of $G$. Each row of $P - Y$ adds up to $1 - 1 = 0$, which is why $\mathbf{b}$'s
gradient does too: adding the same number to every class's bias changes no probability.